# Advanced RAG Systems #1 — Hybrid Search + Reranking

Naive vector search vs the 2026 production playbook, on the same tiny knowledge base.

**The question:** *What is the SLA (Service Level Agreement) refund percentage for enterprise downtime exceeding 4 hours?*

Run the cells top to bottom (Runtime → Run all). Every code line is commented.

In [ ]:
# One-time setup: install the two libraries this demo needs.
# sentence-transformers = embedding + reranking models; rank-bm25 = keyword search.
# The "!" means: run this as a shell command, not as Python.
!pip install -q sentence-transformers rank-bm25

## The corpus — 11 chunks, with traps

Three plans with three different percentages (10% / 15% / 25%), an FAQ that talks *about* refunds but contains no number, and an overview page full of the right words with no answer.

In [ ]:
# The knowledge base: 11 chunks (passages) from a fictional Nimbus Cloud wiki.
# Each entry is (chunk_name, chunk_text). Traps are built in on purpose:
# three plans with three percentages (10% / 15% / 25%), an FAQ that talks about
# refunds but has NO number, and an overview page with the right words, no answer.
CORPUS = [
    ("sla_standard",
     "Nimbus Cloud SLA: Standard plan customers receive a 10% service credit when monthly "
     "downtime exceeds 1 hour. Credits are applied automatically to the next invoice. "
     "The Standard plan SLA covers compute and storage services only."),
    ("sla_enterprise",   # <-- the chunk that actually answers the question (25%)
     "Nimbus Cloud SLA: Enterprise tier customers receive a 25% service credit when monthly "
     "downtime exceeds 4 hours. The Enterprise SLA covers compute, storage, and networking. "
     "Claims must be filed within 30 days of the incident."),
    ("enterprise_support",
     "Enterprise support includes 24/7 phone support with a 15-minute response time for "
     "critical issues. Enterprise customers get a dedicated account manager and quarterly "
     "business reviews."),
    ("refund_policy",
     "Refund policy: monthly plans can be refunded within 7 days of purchase, annual plans "
     "within 30 days. Enterprise contracts are non-refundable except as stated in the SLA."),
    ("sla_pro",
     "Nimbus Cloud SLA: Pro plan customers receive a 15% service credit when monthly "
     "downtime exceeds 2 hours. Credits are applied automatically to the next invoice."),
    ("downtime_definition",
     "Downtime is measured as minutes when the service returns 5xx errors for over 60 "
     "consecutive seconds, excluding scheduled maintenance windows announced 72 hours in advance."),
    ("status_page",
     "Check status.nimbus.example for real-time availability. Historical uptime for the last "
     "90 days is published there with per-region breakdowns."),
    ("credits_process",
     "Service credits are applied to the next invoice automatically. Credits cannot be "
     "converted to cash refunds under any plan."),
    ("sla_overview",   # trap: right words ("enterprise", "SLA", "service credits"), no percentage
     "Nimbus Cloud's SLA program reflects our commitment to reliability. Every plan includes "
     "service level objectives for uptime, measured monthly. Enterprise customers enjoy "
     "enhanced SLA terms with higher service credits for extended outages. See the "
     "plan-specific pages for exact percentages."),
    ("outage_postmortem",
     "Postmortem: the March 3 enterprise outage lasted 4.5 hours and SLA credits were issued "
     "per policy. Root cause was a load balancer misconfiguration. No action needed."),
    ("sla_faq",   # trap: repeats the question's words, contains no percentage at all
     "SLA FAQ: what is an SLA refund? Service credits compensate customers for downtime. "
     "How do I claim? File a claim within 30 days. Which plans are covered? All paid plans "
     "include an SLA."),
]

# The question we will ask the system.
QUERY = "What is the SLA refund percentage for enterprise downtime exceeding 4 hours?"
# Which chunk SHOULD rank first, so the notebook can grade itself.
CORRECT_ID = "sla_enterprise"

## Step 1 — Naive retrieval: embed, cosine, top-3

One embedding model turns every chunk and the question into vectors. Cosine similarity ranks the chunks. Watch which chunk wins — and whether it contains the answer.

In [ ]:
# Import the embedding model class and numpy (fast number-crunching).
from sentence_transformers import SentenceTransformer
import numpy as np

# Load the embedding model. First run downloads ~90 MB of weights, then it is cached.
# all-MiniLM-L6-v2: small, free, runs on CPU. Output: one 384-number vector per text.
emb = SentenceTransformer("all-MiniLM-L6-v2")

# Keep only the chunk texts (drop the names) - the model reads text, not labels.
texts = [t for _, t in CORPUS]

# Embed all 11 chunks: 11 vectors, 384 numbers each.
# normalize_embeddings=True rescales each vector to length 1, so that the plain
# dot product below IS the cosine similarity (no division by lengths needed).
E = emb.encode(texts, normalize_embeddings=True)

# Embed the question the same way: 1 vector of 384 numbers.
q = emb.encode([QUERY], normalize_embeddings=True)[0]

# E @ q = matrix multiplication: dot product of q against EACH of the 11 rows.
# Result: 11 cosine-similarity scores, one per chunk. Higher = closer in meaning.
cos = E @ q

# Sort chunk indexes by score, highest first; keep the top 3.
naive = list(np.argsort(-cos))[:3]

# Print the naive ranking and mark the chunk that contains the real answer.
for r, i in enumerate(naive):
    cid, _ = CORPUS[i]
    mark = "  <-- CORRECT (25%)" if cid == CORRECT_ID else ""
    print(f"#{r+1} {cid:20s} score={cos[i]:.4f}{mark}")

**The failure:** the FAQ wins on meaning-similarity (it repeats the question's own words) while containing no percentage. The correct chunk sits at #2. Retrieval quality is the ceiling — the LLM can only stand on what it is handed.

## Step 2 — Advanced: hybrid + fusion + rerank

1. **BM25** (Best Matching 25) — keyword search: exact words, rare words weighted more.
2. **Dense** — the same vector search as Step 1.
3. **RRF** (Reciprocal Rank Fusion) — merge the two ranked lists: a chunk scores `1/(60 + its rank)` in each list, summed.
4. **Cross-encoder rerank** — a small model reads the question *together with* each candidate chunk and re-scores. A question asking 'what is the percentage' needs a chunk containing a percentage.

In [ ]:
# BM25Okapi = keyword search (Best Matching 25). CrossEncoder = the reranking judge.
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

def rrf(rank_lists, k=60):
    # Reciprocal Rank Fusion: merge several ranked lists into one.
    # A chunk earns 1 / (k + its position) from each list it appears in.
    # Rank-based (not score-based), so the two search methods' different
    # score scales never need to be reconciled. k=60 is the standard constant.
    scores = {}
    for ranks in rank_lists:                 # for each ranked list (BM25, dense)
        for rank, idx in enumerate(ranks):   # rank 0 = first place, 1 = second, ...
            scores[idx] = scores.get(idx, 0) + 1.0 / (k + rank + 1)
    # Return chunk indexes sorted by fused score, best first.
    return sorted(scores, key=scores.get, reverse=True)

# Split every chunk into lowercase words - BM25 matches exact words, nothing fancier.
tok = [t.lower().split() for t in texts]
# Build the keyword index over all 11 chunks.
bm25 = BM25Okapi(tok)
# Score every chunk against the question's words (rare words count more).
b = bm25.get_scores(QUERY.lower().split())
# Keyword search's top 8 chunk indexes.
top_bm25 = list(np.argsort(-b))[:8]
# Vector search's top 8 (reuses the cosine scores "cos" from the naive cell).
top_dense = list(np.argsort(-cos))[:8]
# Fuse the two top-8 lists into one candidate list of 8.
fused = rrf([top_bm25, top_dense])[:8]

# Load the reranker: a small model that reads a (question, chunk) pair TOGETHER
# and outputs one relevance score - slower than vector search, far more accurate.
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
# Score each of the 8 fused candidates against the question.
rs = reranker.predict([(QUERY, texts[i]) for i in fused])
# Re-sort the candidates by the reranker's scores, best first.
reranked = [fused[i] for i in np.argsort(-rs)]

# Print the advanced top 3 and mark the correct chunk.
for r, i in enumerate(reranked[:3]):
    cid, _ = CORPUS[i]
    mark = "  <-- CORRECT (25%)" if cid == CORRECT_ID else ""
    print(f"#{r+1} {cid:20s} rerank score={rs[list(fused).index(i)]:.4f}{mark}")

# Show the winning chunk in full - this is literally what would be pasted
# into the LLM's prompt as evidence.
print("\nTop chunk handed to the LLM:")
print(CORPUS[reranked[0]][1])

## Takeaway

Same corpus, same question — the retrieval stack was the entire difference. Naive RAG (chunk → embed → top-k) is the floor; production RAG is a portfolio of techniques, each fixing one failure mode of the last.

*Inside the LLM — Advanced RAG Systems #1*